# NexaTel Customer Churn Analytics Phase 3: KPI Calculations

This notebook calculates the 10 management KPIs requested in the PRD and includes the optional cohort retention analysis. Every KPI is labelled with its formula, result, and a short business interpretation.

**Assumptions:** This is a historical customer snapshot, not a complete monthly event ledger. Therefore, the overall churn rate uses the available customer base as the starting-base proxy. ARPU uses active subscription recurring charges. Gross margin is set as an editable assumption because no margin column is present in the source files.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

PROJECT = Path(r"E:\NexaTel-Customer")
REPORTS = PROJECT / "phase3_reports"
REPORTS.mkdir(exist_ok=True)

def file_path(name):
    clean = PROJECT / "cleaned_data" / f"{name}_clean.csv"
    root = PROJECT / f"{name}.csv"
    folder = PROJECT / "csv_files" / f"{name}.csv"
    return clean if clean.exists() else root if root.exists() else folder

def load(name):
    return pd.read_csv(file_path(name), dtype="string")

customers = load("customers").drop_duplicates("customer_id").copy()
subscriptions = load("subscriptions").drop_duplicates("subscription_id").copy()
tickets = load("support_tickets").copy()
contracts = load("contracts").copy()
plans = load("plans").drop_duplicates("plan_id").copy()

for df, columns in [(customers, ["age", "tenure_months", "annual_income_inr"]), (subscriptions, ["monthly_charge_inr"]), (plans, ["monthly_charge"]), (tickets, ["resolution_hours"]), (contracts, ["contract_length_months"])]:
    for column in columns:
        if column in df:
            df[column] = pd.to_numeric(df[column], errors="coerce")

for df, columns in [(customers, ["acquisition_date", "churn_date"]), (subscriptions, ["start_date", "end_date"]), (tickets, ["created_date", "resolution_date"]), (contracts, ["start_date", "end_date"])]:
    for column in columns:
        df[column] = pd.to_datetime(df[column], errors="coerce", dayfirst=True)

customers["churned"] = customers["customer_status"].str.strip().str.lower().eq("churned")
subscriptions["active"] = subscriptions["subscription_status"].str.strip().str.lower().eq("active")
tickets["resolved"] = tickets["status"].str.strip().str.lower().eq("resolved")
contracts["renewed"] = contracts["renewal_status"].str.strip().str.lower().eq("renewed")

# Editable assumption because the source data has no gross-margin field.
GROSS_MARGIN = 0.70
print("Loaded KPI tables successfully.")

C:\Users\Admin\AppData\Local\Temp\ipykernel_17276\2727024472.py:32: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df[column] = pd.to_datetime(df[column], errors="coerce", dayfirst=True)
C:\Users\Admin\AppData\Local\Temp\ipykernel_17276\2727024472.py:32: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df[column] = pd.to_datetime(df[column], errors="coerce", dayfirst=True)
C:\Users\Admin\AppData\Local\Temp\ipykernel_17276\2727024472.py:32: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df[column] = pd.to_datetime(df[column], errors="coerce", dayfirst=True)


Loaded KPI tables successfully.


## KPI 1  Customer Churn Rate

**Formula:** Customers Lost / Customers at Start ? 100

For this snapshot, Customers at Start is represented by the available customer base, because there is no separate monthly opening-base table.

In [2]:
customers_lost = int(customers["churned"].sum())
customers_at_start = len(customers)
churn_rate = customers_lost / customers_at_start * 100
print(f"Customer Churn Rate = {customers_lost:,} / {customers_at_start:,} ? 100 = {churn_rate:.2f}%")
print("Interpretation: Compare this headline rate with the PRD target of less than 2.1% monthly. This snapshot rate should not be called a monthly rate without monthly opening and closing bases.")

Customer Churn Rate = 3,506 / 19,000 ? 100 = 18.45%
Interpretation: Compare this headline rate with the PRD target of less than 2.1% monthly. This snapshot rate should not be called a monthly rate without monthly opening and closing bases.


## KPI 2  Customer Retention Rate

**Formula:** (End - New) / Start ? 100

Because a monthly new-customer field is not available, this snapshot uses the complementary rate: active customers / total starting customer base ? 100.

In [3]:
active_customers = int((~customers["churned"]).sum())
retention_rate = active_customers / customers_at_start * 100
print(f"Customer Retention Rate = {active_customers:,} / {customers_at_start:,} ? 100 = {retention_rate:.2f}%")

Customer Retention Rate = 15,494 / 19,000 ? 100 = 81.55%


## KPI 3  ARPU

**Formula:** Total Recurring Revenue / Average Active Subscribers

The available recurring charge is `monthly_charge_inr` in active subscriptions.

In [4]:
active_subs = subscriptions[subscriptions["active"]].copy()
total_recurring_revenue = active_subs["monthly_charge_inr"].sum()
avg_active_subscribers = active_subs["customer_id"].nunique()
arpu = total_recurring_revenue / avg_active_subscribers if avg_active_subscribers else np.nan
print(f"ARPU = INR {total_recurring_revenue:,.2f} / {avg_active_subscribers:,} active subscribers = INR {arpu:,.2f}")

ARPU = INR 14,920,323.00 / 15,674 active subscribers = INR 951.92


## KPI 4  Customer Lifetime Value

**Formula:** ARPU ? Gross Margin % ? (1 / Churn Rate)

Gross margin is an editable assumption of 70% because the source data does not contain a gross-margin field.

In [5]:
clv = arpu * GROSS_MARGIN * (1 / (churn_rate / 100)) if churn_rate else np.nan
print(f"CLV = INR {arpu:,.2f} ? {GROSS_MARGIN:.0%} ? (1 / {churn_rate / 100:.4f}) = INR {clv:,.2f}")
print("Interpretation: CLV is sensitive to the assumed gross margin and churn definition; update GROSS_MARGIN when Finance provides the approved value.")

CLV = INR 951.92 ? 70% ? (1 / 0.1845) = INR 3,611.09
Interpretation: CLV is sensitive to the assumed gross margin and churn definition; update GROSS_MARGIN when Finance provides the approved value.


## KPI 5  Revenue Lost to Churn

**Formula:** Churned Customers ? Their ARPU

For this snapshot, churned customers are assigned the average monthly charge of their subscription where available.

In [6]:
churned_with_charge = customers[["customer_id", "churned"]].merge(subscriptions[["customer_id", "monthly_charge_inr"]], on="customer_id", how="left")
churned_with_charge = churned_with_charge[churned_with_charge["churned"]]
revenue_lost = churned_with_charge["monthly_charge_inr"].mean() * len(churned_with_charge)
print(f"Revenue Lost to Churn = {len(churned_with_charge):,} churned customers ? average charge = INR {revenue_lost:,.2f} per month")

Revenue Lost to Churn = 3,757 churned customers ? average charge = INR 2,566,181.00 per month


## KPI 6  Monthly Recurring Revenue

**Formula:** Sum of active recurring charges.

In [7]:
mrr = active_subs["monthly_charge_inr"].sum()
print(f"Monthly Recurring Revenue = INR {mrr:,.2f}")

Monthly Recurring Revenue = INR 14,920,323.00


## KPI 7  First Contact Resolution

**Formula:** Resolved on First Contact / Total Tickets ? 100

The source field used is `first_contact_resolution`.

In [8]:
fcr_yes = tickets["first_contact_resolution"].str.strip().str.lower().eq("yes").sum()
fcr_rate = fcr_yes / len(tickets) * 100
print(f"First Contact Resolution = {fcr_yes:,} / {len(tickets):,} ? 100 = {fcr_rate:.2f}%")
print("Target comparison: " + ("above" if fcr_rate > 78 else "below") + " the PRD target of 78%.")

First Contact Resolution = 12,726 / 31,490 ? 100 = 40.41%
Target comparison: below the PRD target of 78%.


## KPI 8  Average Resolution Time

**Formula:** Mean resolution hours for resolved tickets

The source already provides `resolution_hours`; unresolved tickets are excluded.

In [9]:
resolved_hours = tickets.loc[tickets["resolved"], "resolution_hours"].dropna()
avg_resolution_time = resolved_hours.mean()
print(f"Average Resolution Time = {avg_resolution_time:.2f} hours")
print("Target comparison: " + ("below" if avg_resolution_time < 24 else "above") + " the PRD target of 24 hours.")

Average Resolution Time = 16.04 hours
Target comparison: below the PRD target of 24 hours.


## KPI 9  Contract Renewal Rate

**Formula:** Contracts Renewed / Due for Renewal ? 100

Contracts with a non-null `end_date` are treated as due for renewal; `renewal_status = Renewed` counts as renewed.

In [10]:
due_contracts = contracts[contracts["end_date"].notna()]
renewed_contracts = due_contracts[due_contracts["renewed"]]
renewal_rate = len(renewed_contracts) / len(due_contracts) * 100 if len(due_contracts) else np.nan
print(f"Contract Renewal Rate = {len(renewed_contracts):,} / {len(due_contracts):,} ? 100 = {renewal_rate:.2f}%")
print("Target comparison: " + ("above" if renewal_rate > 88 else "below") + " the PRD target of 88%.")

Contract Renewal Rate = 3,676 / 10,607 ? 100 = 34.66%
Target comparison: below the PRD target of 88%.


## KPI 10  Average Customer Tenure

**Formula:** Mean `tenure_months` of the active base.

In [11]:
active_customer_ids = set(customers.loc[~customers["churned"], "customer_id"])
avg_tenure = customers[customers["customer_id"].isin(active_customer_ids)]["tenure_months"].mean()
print(f"Average Customer Tenure = {avg_tenure:.2f} months")

Average Customer Tenure = 87.08 months


## Optional bonus  Cohort retention analysis

Customers are grouped by acquisition month. A customer is counted as retained at 3, 6, or 12 months when they have no churn date before that milestone.

In [12]:
cohort = customers.dropna(subset=["acquisition_date"]).copy()
cohort["cohort_month"] = cohort["acquisition_date"].dt.to_period("M").astype(str)
for months in [3, 6, 12]:
    milestone = cohort["acquisition_date"] + pd.DateOffset(months=months)
    cohort[f"active_after_{months}m"] = cohort["churn_date"].isna() | (cohort["churn_date"] >= milestone)
cohort_retention = cohort.groupby("cohort_month")[["active_after_3m", "active_after_6m", "active_after_12m"]].mean().mul(100).round(2).reset_index()
cohort_retention.to_csv(REPORTS / "cohort_retention.csv", index=False)
display(cohort_retention.head(15))

,cohort_month,active_after_3m,active_after_6m,active_after_12m
0,2014-01,99.85,99.66,99.31
1,2014-02,100.00,100.00,100.00
2,2014-03,100.00,100.00,100.00
3,2014-04,100.00,100.00,100.00
4,2014-05,98.92,98.92,98.92
5,2014-06,100.00,100.00,99.11
6,2014-07,99.12,98.25,98.25
7,2014-08,99.07,99.07,99.07
8,2014-09,99.10,99.10,98.20
9,2014-10,100.00,100.00,99.16


## KPI summary

The following table contains all required KPIs and is saved as `kpi_summary.csv`.

In [13]:
kpi_summary = pd.DataFrame([
    ["Customer Churn Rate", churn_rate, "%", "Customers lost / customers at start"],
    ["Customer Retention Rate", retention_rate, "%", "Active customers / starting base proxy"],
    ["ARPU", arpu, "INR/month", "Recurring revenue / active subscribers"],
    ["Customer Lifetime Value", clv, "INR", "ARPU ? gross margin ? 1/churn rate"],
    ["Revenue Lost to Churn", revenue_lost, "INR/month", "Churned customers ? their average charge"],
    ["Monthly Recurring Revenue", mrr, "INR/month", "Sum of active recurring charges"],
    ["First Contact Resolution", fcr_rate, "%", "FCR yes / total tickets"],
    ["Average Resolution Time", avg_resolution_time, "hours", "Mean resolution hours for resolved tickets"],
    ["Contract Renewal Rate", renewal_rate, "%", "Renewed / due contracts"],
    ["Average Customer Tenure", avg_tenure, "months", "Mean active customer tenure"]
], columns=["KPI", "Value", "Unit", "Formula used"])
kpi_summary.to_csv(REPORTS / "kpi_summary.csv", index=False)
display(kpi_summary)
print("Phase 3 completed. Reports saved to:", REPORTS)

,KPI,Value,Unit,Formula used
0,Customer Churn Rate,1.845263e+01,%,Customers lost / customers at start
1,Customer Retention Rate,8.154737e+01,%,Active customers / starting base proxy
2,ARPU,9.519155e+02,INR/month,Recurring revenue / active subscribers
3,Customer Lifetime Value,3.611088e+03,INR,ARPU ? gross margin ? 1/churn rate
4,Revenue Lost to Churn,2.566181e+06,INR/month,Churned customers ? their average charge
5,Monthly Recurring Revenue,1.492032e+07,INR/month,Sum of active recurring charges
6,First Contact Resolution,4.041283e+01,%,FCR yes / total tickets
7,Average Resolution Time,1.604447e+01,hours,Mean resolution hours for resolved tickets
8,Contract Renewal Rate,3.465636e+01,%,Renewed / due contracts
9,Average Customer Tenure,8.707538e+01,months,Mean active customer tenure


Phase 3 completed. Reports saved to: E:\NexaTel-Customer\phase3_reports
